In [ ]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import itertools
import json
from scipy.stats import sem
import scipy

In [ ]:
num_class = [47, 10, 43, 10, 45, 196, 397, 10]
dataset_name = ["DTD", "EuroSAT", "GTSRB", "MNIST", "RESISC45", "Stanford_Cars", "SUN397", "SVHN"]
domain = "Base_Fine_Tuned"
transform_type = "Standard" # Base_Fine_Tuned_Classifier | Standard
model_name = "CLIP_ViT_Vision"
refining_type = "Standard" # Increment_Training | Standard
refiner = "Fine_Tuned" # Fine_Tuned | Reverse_Probe
indices = [i for i in range(12)]

In [ ]:
# Store all combinations of lengths 2 to 8
all_combinations = {}

for r in range(1, 9):  # lengths 2 through 8
    combos = list(itertools.combinations(dataset_name, r))
    all_combinations[r] = combos

## Loading of Matrices

### SVD Frobenius Norm

In [ ]:
task_matrix = {}

for num_sets in range(1,9):
    task_matrix[num_sets] = {}
    for combo in range(len(all_combinations[num_sets])):
        matrix_name = ""
        for c in all_combinations[num_sets][combo]:
            matrix_name += f"{c}_"
        W_folder = f"../Data/Multi_Class_Augmentation/{refining_type}/{refiner}/Task_Matrices/{num_sets}/{domain}/{matrix_name}"
        task_matrix[num_sets][matrix_name] = {}
        for rep in range(1,6):
            task_matrix[num_sets][matrix_name][rep] = {"U": {}, "S": {}, "Vh": {}}
            W = np.load(f"{W_folder}/{rep}.npy")
            for i in indices:
                task_matrix[num_sets][matrix_name][rep]["U"][i], task_matrix[num_sets][matrix_name][rep]["S"][i], task_matrix[num_sets][matrix_name][rep]["Vh"][i] = np.linalg.svd(W[i])

In [ ]:
energy_capture = {}

for num_sets in range(1,9):
    energy_capture[num_sets] = {}
    for combo in range(len(all_combinations[num_sets])):
        matrix_name = ""
        for c in all_combinations[num_sets][combo]:
            matrix_name += f"{c}_"
        energy_capture[num_sets][matrix_name] = {}
        for rep in range(1,6):
            energy_capture[num_sets][matrix_name][rep] = {}
            for i in indices:
                energy = task_matrix[num_sets][matrix_name][rep]["S"][i] ** 2
                cumulative = np.cumsum(energy)
                total = cumulative[-1]
                frac = cumulative / total
                threshold = 0.95
                k = np.searchsorted(frac, threshold) + 1
                energy_capture[num_sets][matrix_name][rep][i] = k

In [ ]:
import copy

copier = copy.deepcopy(energy_capture)

for num_sets in range(1,9):
    for combo in range(len(all_combinations[num_sets])):
        matrix_name = ""
        for c in all_combinations[num_sets][combo]:
            matrix_name += f"{c}_"
        for rep in range(1,6):
            for i in indices:
                copier[num_sets][matrix_name][rep][i] = int(copier[num_sets][matrix_name][rep][i])

In [ ]:
import json

with open("k_95%_Task_Matrices.json", "w") as f:
    json.dump(copier, f, indent=2)

### Rank

In [ ]:
task_matrix_rank = {}

for num_sets in range(1,9):
    task_matrix_rank[num_sets] = {}
    for combo in range(len(all_combinations[num_sets])):
        matrix_name = ""
        for c in all_combinations[num_sets][combo]:
            matrix_name += f"{c}_"
        W_folder = f"../Data/Multi_Class_Augmentation/{refining_type}/{refiner}/Task_Matrices/{num_sets}/{domain}/{matrix_name}"
        task_matrix_rank[num_sets][matrix_name] = {}
        for rep in range(1,6):
            task_matrix_rank[num_sets][matrix_name][rep] = {}
            W = np.load(f"{W_folder}/{rep}.npy")
            for i in indices:
                task_matrix_rank[num_sets][matrix_name][rep][i] = np.linalg.matrix_rank(W[i])

In [ ]:
import copy

copier = copy.deepcopy(task_matrix_rank)

for num_sets in range(1,9):
    for combo in range(len(all_combinations[num_sets])):
        matrix_name = ""
        for c in all_combinations[num_sets][combo]:
            matrix_name += f"{c}_"
        for rep in range(1,6):
            for i in indices:
                copier[num_sets][matrix_name][rep][i] = int(copier[num_sets][matrix_name][rep][i])

In [ ]:
import json

with open("Task_Matrices_Rank.json", "w") as f:
    json.dump(copier, f, indent=2)

### Eigen Values & Vectors

In [ ]:
task_matrix_eigen_val = {}

for num_sets in range(1,9):
    task_matrix_rank[num_sets] = {}
    for combo in range(len(all_combinations[num_sets])):
        matrix_name = ""
        for c in all_combinations[num_sets][combo]:
            matrix_name += f"{c}_"
        W_folder = f"../Data/Multi_Class_Augmentation/{refining_type}/{refiner}/Task_Matrices/{num_sets}/{domain}/{matrix_name}"
        task_matrix_rank[num_sets][matrix_name] = {}
        for rep in range(1,6):
            task_matrix_rank[num_sets][matrix_name][rep] = {}
            W = np.load(f"{W_folder}/{rep}.npy")
            for i in indices:
                task_matrix_rank[num_sets][matrix_name][rep][i] = np.linalg.matrix_rank(W[i])

In [ ]:
matrix_name = "EuroSAT_"
num_sets = 1
W_folder = f"../Data/Multi_Class_Augmentation/{refining_type}/{refiner}/Task_Matrices/{num_sets}/{domain}/{matrix_name}"
W = np.load(f"{W_folder}/1.npy")

In [ ]:
eigenvalues, eigenvectors = np.linalg.eig(W[11])

In [ ]:
idx = np.argsort(-np.abs(eigenvalues))  # descending by magnitude
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

In [ ]:
import matplotlib.pyplot as plt

plt.plot(np.abs(eigenvalues))
plt.title("Eigenvalue Magnitudes")
plt.xlabel("Index")
plt.ylabel("|λ|")
plt.show()

In [ ]:
explained_variance = np.cumsum(np.abs(eigenvalues)) / np.sum(np.abs(eigenvalues))
print(explained_variance)

In [ ]:
k = np.argmax(explained_variance >= 0.95)
print(k)

### Condition Number

In [ ]:
task_matrix_condition_num = {}

for num_sets in range(1,9):
    task_matrix_condition_num[num_sets] = {}
    for combo in range(len(all_combinations[num_sets])):
        matrix_name = ""
        for c in all_combinations[num_sets][combo]:
            matrix_name += f"{c}_"
        W_folder = f"../Data/Multi_Class_Augmentation/{refining_type}/{refiner}/Task_Matrices/{num_sets}/{domain}/{matrix_name}"
        task_matrix_condition_num[num_sets][matrix_name] = {}
        for rep in range(1,6):
            task_matrix_condition_num[num_sets][matrix_name][rep] = {}
            W = np.load(f"{W_folder}/{rep}.npy")
            for i in indices:
                task_matrix_condition_num[num_sets][matrix_name][rep][i] = np.linalg.cond(W[i])

In [ ]:
import copy

copier = copy.deepcopy(task_matrix_condition_num)

for num_sets in range(1,9):
    for combo in range(len(all_combinations[num_sets])):
        matrix_name = ""
        for c in all_combinations[num_sets][combo]:
            matrix_name += f"{c}_"
        for rep in range(1,6):
            for i in indices:
                copier[num_sets][matrix_name][rep][i] = int(copier[num_sets][matrix_name][rep][i])

In [ ]:
import json

with open("Task_Matrices_Condition_Number.json", "w") as f:
    json.dump(copier, f, indent=2)

### Operator Norm

In [ ]:
task_matrix_operator_norm = {}

for num_sets in range(1,9):
    task_matrix_operator_norm[num_sets] = {}
    for combo in range(len(all_combinations[num_sets])):
        matrix_name = ""
        for c in all_combinations[num_sets][combo]:
            matrix_name += f"{c}_"
        W_folder = f"../Data/Multi_Class_Augmentation/{refining_type}/{refiner}/Task_Matrices/{num_sets}/{domain}/{matrix_name}"
        task_matrix_operator_norm[num_sets][matrix_name] = {}
        for rep in range(1,6):
            task_matrix_operator_norm[num_sets][matrix_name][rep] = {}
            W = np.load(f"{W_folder}/{rep}.npy")
            for i in indices:
                task_matrix_operator_norm[num_sets][matrix_name][rep][i] = np.linalg.norm(W[i], ord=2)

In [ ]:
import copy

copier = copy.deepcopy(task_matrix_operator_norm)

for num_sets in range(1,9):
    for combo in range(len(all_combinations[num_sets])):
        matrix_name = ""
        for c in all_combinations[num_sets][combo]:
            matrix_name += f"{c}_"
        for rep in range(1,6):
            for i in indices:
                copier[num_sets][matrix_name][rep][i] = int(copier[num_sets][matrix_name][rep][i])

In [ ]:
import json

with open("Task_Matrices_Operator_Norm.json", "w") as f:
    json.dump(copier, f, indent=2)

### Determinant - Note Can't do because of Overflow and Numerical Instability

In [ ]:
task_matrix_determinant = {}

for num_sets in range(1,9):
    task_matrix_determinant[num_sets] = {}
    for combo in range(len(all_combinations[num_sets])):
        matrix_name = ""
        for c in all_combinations[num_sets][combo]:
            matrix_name += f"{c}_"
        W_folder = f"../Data/Multi_Class_Augmentation/{refining_type}/{refiner}/Task_Matrices/{num_sets}/{domain}/{matrix_name}"
        task_matrix_determinant[num_sets][matrix_name] = {}
        for rep in range(1,6):
            task_matrix_determinant[num_sets][matrix_name][rep] = {}
            W = np.load(f"{W_folder}/{rep}.npy")
            for i in indices:
                task_matrix_determinant[num_sets][matrix_name][rep][i] = np.linalg.det(W[i])

In [ ]:
import copy

copier = copy.deepcopy(task_matrix_determinant)

for num_sets in range(1,9):
    for combo in range(len(all_combinations[num_sets])):
        matrix_name = ""
        for c in all_combinations[num_sets][combo]:
            matrix_name += f"{c}_"
        for rep in range(1,6):
            for i in indices:
                copier[num_sets][matrix_name][rep][i] = int(copier[num_sets][matrix_name][rep][i])

In [ ]:
import json

with open("Task_Matrices_Determinant.json", "w") as f:
    json.dump(copier, f, indent=2)

## Graphing

### SVD Frobenius Norm

In [ ]:
with open(f"./{refining_type}/{refiner}/Task_Matrices_Operator_Norm.json", "r") as f:
    data = json.load(f)

In [ ]:
for i in range(1,9):
    for combo in range(len(all_combinations[i])):
        matrix_name = ""
        for c in all_combinations[i][combo]:
            matrix_name += f"{c}_"
        temp_arr = []
        for rep in range(1,6):
            temp_arr.append(data[str(i)][matrix_name][str(rep)][str(11)])
        ci_low, ci_high = scipy.stats.t.interval(0.95, df=len(temp_arr)-1, loc=np.mean(temp_arr), scale=sem(temp_arr))
        print(f"{matrix_name}: {np.mean(temp_arr)} +- {np.mean(temp_arr)-ci_low}")

### Rank Analysis

In [ ]:
with open(f"./{refining_type}/{refiner}/Task_Matrices_Rank.json", "r") as f:
    data = json.load(f)

In [ ]:
for i in range(1,9):
    for combo in range(len(all_combinations[i])):
        matrix_name = ""
        for c in all_combinations[i][combo]:
            matrix_name += f"{c}_"
        temp_arr = []
        for rep in range(1,6):
            temp_arr.append(data[str(i)][matrix_name][str(rep)][str(11)])
        ci_low, ci_high = scipy.stats.t.interval(0.95, df=len(temp_arr)-1, loc=np.mean(temp_arr), scale=sem(temp_arr))
        print(f"{matrix_name}: {np.mean(temp_arr)} +- {np.mean(temp_arr)-ci_low}")

## Old

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(S[11][6], marker='.', linewidth=1)
plt.title('Singular values (linear scale)')
plt.xlabel('index i')
plt.ylabel(r'$\sigma_i$')
plt.grid(True)

plt.figure(figsize=(8,4))
plt.semilogy(S[11][6], marker='.', linewidth=1)
plt.title('Singular values (log scale)')
plt.xlabel('index i')
plt.ylabel(r'$\sigma_i$ (log scale)')
plt.grid(True)
plt.show()

In [ ]:
energy = S[11][6]**2
cumulative = np.cumsum(energy)
total = cumulative[-1]
frac = cumulative / total  # fraction of Frobenius energy captured

# Pick k for a threshold
threshold = 0.95
k = np.searchsorted(frac, threshold) + 1  # +1 because searchsorted returns idx
print("k for {:.0%} energy:".format(threshold), k)

# Plot fraction
plt.figure(figsize=(8,4))
plt.plot(frac, linewidth=2)
plt.axhline(threshold, color='red', linestyle='--')
plt.xlabel('k')
plt.ylabel('Fraction of energy captured')
plt.grid(True)
plt.show()

In [ ]:
# Operator Norm
print(np.linalg.norm(S[11][6], ord=2))